# FINE slide figures

These panels are the FINE results in `src/non_sep_etas.pdf`: cumulative counts, magnitude and joint log-likelihood, information gain, the Poisson and negative-binomial number tests, the binary spatial test and the S-test, and the PL, N, M, and S catalog tests.

Short horizons (0.5 / 1 / 4 d) load `outputs/rolling_continuation_hauksson_pre_ridgecrest_short_v2` (FeatureState sync fix). Long horizons (7 / 60 / 300 d) stay on `outputs/rolling_continuation_hauksson_pre_ridgecrest`. The pre-fix short tree is archived via `notebooks/fine_slides_figures_flawed_short_featurestate.ipynb`.

`STYLE` is the only place that sets fonts, colors, sizes, and the save switches. Run the notebook after editing it. `save_png`, `save_pdf`, and `save_svg` write every panel under `outputs/fine_slides_figures`. `SLIDE_HORIZON` is the forecast length on the single-horizon slides.

The PL, N, M, and S panels are the mean of each per-catalog statistic, with a shaded ±1 standard-deviation band. They read `PL_mean`, `N_mean`, `M_mean`, `S_mean` and the matching `*_std` columns from the catalog-test cache.


In [ ]:
from __future__ import annotations

%matplotlib inline

import io
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# ---------------------------------------------------------------------------
# Style — change these, then re-run the notebook. Do not hard-code colors below.
# ---------------------------------------------------------------------------
STYLE = {
    "font": {
        "family": "DejaVu Sans",
        "size": 13,
        "title": 12,
        "label": 13,
        "tick": 12,
        "legend": 11,
    },
    "colors": {
        "observed": "#1a1a1a",
        "expected": "#111111",
        "poisson": "#111111",
        "nbd": "#737373",
        "reference": "#808080",
        "grid": "#d0d0d0",
        "pass": "#284473",
        "fail": "#001AFF",
        "secondary": {
            "pass": "#995062",
            "fail": "#FF0000",
        },
        "magnitude": "#6e2a2a",
        "methods": {
            "etas": "#284473",
            "FINE": "#E6225B",
        },
    },
    "cmaps": {
        "discrepancy": "coolwarm",
        "coverage": "Greys",
        "rate": "viridis",
        "spatial_observed": "Greys",
        "spatial_forecast": "viridis",
    },
    "labels": {"etas": "ETAS", "FINE": "FINE"},
    "markers": {"etas": "o", "FINE": "s"},
    "scatter_markers": {"etas": "s", "FINE": "o"},
    "scatter_hollow": {"etas": True, "FINE": False},
    "cumulative_counts_aspect": [5, 3],
    "across_horizons_aspect": [5, 3],
    "quantile_series_aspect": [5, 3],
    "cumulative_igpa_aspect": [5, 3],
    "figsize": {
        "metric": (7.2, 4.2),
        "wide": (11.0, 4.4),
        "cumulative_counts": (11.0, 6.6),
        "quantile_series": (11.0, 6.6),
        "cumulative_igpa": (11.0, 6.6),
        "wide_short": (11.0, 3.6),
        "square": (6.4, 6.4),
        "hist": (5.2, 3.6),
        "map": (6.4, 5.6),
        "number_rows": (7.2, 3.4),
    },
    "line_width": 1.6,
    "marker_size": 4.5,
    "scatter_size": 70,
    "coverage_marker": 28,
    "marker_area_base": 14.0,
    "marker_area_scale": 5.0,
    "count_floor": 0.8,
    "magnitude_size_scale": 6.0,
    "magnitude_size_shift": 2.0,
    "epicenter_scale": 3.5,
    "epicenter_growth": 2.6,
    "spatial_bins": 60,
    "fill_alpha": 0.11,
    "ylabel_labelpad": {"Mean pseudolikelihood": 16},
    "realization_alpha": 0.28,
    "legend_frameon": False,
    "spines_top_right": False,
    "axes_linewidth": 0.8,
    "display_dpi": 110,
    "save_dpi": 300,
    "save_png": True,
    "save_pdf": True,
    "save_svg": True,
    "overwrite": True,
}

# Single-horizon slides in the deck use a 7-day forecast.
SLIDE_HORIZON = 7.0
# None = every cached horizon on the versus-horizon curves.
HORIZONS: list[float] | None = None
METHODS = ["etas", "FINE"]


def find_repo_root() -> Path:
    for start in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (start / "runnable_code" / "presentation_figures.py").is_file():
            return start
    raise FileNotFoundError("Open this notebook from the etas checkout.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "runnable_code") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "runnable_code"))

import importlib
import presentation_figures as figures
importlib.reload(figures)
figures.configure(STYLE)

# FIGURE_DIR = REPO_ROOT / "outputs" / "fine_slides_figures"
FIGURE_DIR = Path("/a/home/cc/students/csguests/neriberman/Repos/etas/outputs/figures_for_statsei14")
# Mean and std of the PL / N / M / S catalog statistics. Written by
# notebooks/presentation_figures.ipynb when COMPUTE_HEAVY_ACROSS is True.
CSEP_MEAN_STD = (
    REPO_ROOT / "outputs" / "presentation_figures" / "across_horizons" / "csep_molchan.csv"
)


def save_figure(figure, filename, directory, dpi=300, save_png=True, save_pdf=True, save_svg=True, overwrite=False):
    """Save a figure as PNG, PDF, and/or SVG."""
    if not os.path.exists(directory):
        os.makedirs(directory)
    if overwrite or not os.path.exists(os.path.join(directory, f"{filename}.png")):
        if save_png:
            figure.savefig(os.path.join(directory, f"{filename}.png"), dpi=dpi, format="png")
    if overwrite or not os.path.exists(os.path.join(directory, f"{filename}.pdf")):
        if save_pdf:
            figure.savefig(os.path.join(directory, f"{filename}.pdf"), dpi=dpi, format="pdf")
    if overwrite or not os.path.exists(os.path.join(directory, f"{filename}.svg")):
        if save_svg:
            figure.savefig(os.path.join(directory, f"{filename}.svg"), dpi=dpi, format="svg")


def _horizon_days_from_folder(folder: tuple[str, ...]) -> float | None:
    if not folder:
        return None
    name = str(folder[0])
    if not (name.startswith("horizon_") and name.endswith("d")):
        return None
    return float(name[len("horizon_"):-1].replace("p", "."))


def _title_names_horizon(title: str, days: float) -> bool:
    text = f"{days:g}"
    return f"{text} d" in title or f"{text}-day" in title


def _with_horizon_title(title: str, days: float) -> str:
    if not title or _title_names_horizon(title, days):
        return title
    suffix = f" — T = {days:g} d"
    if "\n" in title:
        head, tail = title.split("\n", 1)
        return f"{head}{suffix}\n{tail}"
    return f"{title}{suffix}"


def stamp_single_horizon_figure(fig, days: float):
    """Put the forecast length on every title of a one-horizon panel."""
    suptitle = fig._suptitle.get_text() if fig._suptitle is not None else ""
    if suptitle:
        fig.suptitle(_with_horizon_title(suptitle, days))
    for ax in fig.axes:
        title = ax.get_title()
        if title:
            ax.set_title(_with_horizon_title(title, days))
    return fig


def export(fig, filename: str, *folder: str, force: bool = False) -> None:
    """Save one panel and show it. ``folder`` is relative to FIGURE_DIR.

    ``force`` writes png, pdf, and svg and overwrites, even when the STYLE
    save switches are off.
    """
    if fig is None:
        display(Markdown(f"*Skipped `{filename}`.*"))
        return
    days = _horizon_days_from_folder(folder)
    if days is not None:
        stamp_single_horizon_figure(fig, days)
    directory = FIGURE_DIR.joinpath(*folder) if folder else FIGURE_DIR
    save_figure(
        fig,
        filename,
        directory,
        dpi=STYLE["save_dpi"],
        save_png=True if force else STYLE["save_png"],
        save_pdf=True if force else STYLE["save_pdf"],
        save_svg=True if force else STYLE["save_svg"],
        overwrite=True if force else STYLE["overwrite"],
    )
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=STYLE["display_dpi"], bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buffer.getvalue()))


def horizon_tag(days: float) -> str:
    text = f"{days:g}".replace(".", "p")
    return f"horizon_{text}d"


## Cached rolling forecasts

Reads the Hauksson pre-Ridgecrest continuation caches. It does not recompute forecasts.


In [ ]:
import etas.bayona_evaluations as bayona_evaluations
import etas.rolling_analysis as rolling_analysis
from scipy import stats as scipy_stats

EXPERIMENTS = [
    {
        # Post FeatureState-sync fix (replaces contaminated …_short).
        "output_root": REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest_short_v2",
        "horizon_days": [0.5, 1.0, 4.0],
    },
    {
        "output_root": REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest",
        "horizon_days": [7.0, 60.0, 300.0],
    },
]

specs = []
for experiment in EXPERIMENTS:
    root = experiment["output_root"]
    if not root.is_dir():
        continue
    for days in experiment["horizon_days"]:
        horizon_dir = rolling_analysis.horizon_directory(root, float(days))
        cache = horizon_dir / "analysis_cache"
        if (cache / "scores.csv").is_file():
            specs.append({
                "horizon_days": float(days),
                "output_root": root,
                "horizon_dir": horizon_dir,
            })

if HORIZONS is not None:
    chosen = {float(days) for days in HORIZONS}
    specs = [item for item in specs if item["horizon_days"] in chosen]
specs = sorted(specs, key=lambda item: item["horizon_days"])
HORIZON_LIST = [item["horizon_days"] for item in specs]
SLIDE_SPEC = next((item for item in specs if np.isclose(item["horizon_days"], SLIDE_HORIZON)), None)

score_frames = []
summary_frames = []
for experiment in EXPERIMENTS:
    comparison_dir = experiment["output_root"] / "horizon_comparison"
    if (comparison_dir / "scores.csv").is_file():
        score_frames.append(pd.read_csv(comparison_dir / "scores.csv"))
    if (comparison_dir / "summary.csv").is_file():
        summary_frames.append(pd.read_csv(comparison_dir / "summary.csv"))

scores_all = pd.concat(score_frames, ignore_index=True) if score_frames else pd.DataFrame()
summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
if len(scores_all):
    for column in ("forecast_start", "forecast_end"):
        if column in scores_all.columns:
            scores_all[column] = pd.to_datetime(scores_all[column], format="mixed")
    if HORIZONS is not None:
        mask = np.isin(scores_all["horizon_days"].to_numpy(dtype=float), HORIZON_LIST)
        scores_all = scores_all.loc[mask].copy()
        if len(summary):
            mask = np.isin(summary["horizon_days"].to_numpy(dtype=float), HORIZON_LIST)
            summary = summary.loc[mask].copy()

display(Markdown(
    "**Horizons:** "
    + ", ".join(f"{days:g} d" for days in HORIZON_LIST)
    + f"  \n**Slide horizon:** {SLIDE_HORIZON:g} d"
    + (" (cache found)" if SLIDE_SPEC is not None else " (no cache)")
    + f"  \n**Figure directory:** `{FIGURE_DIR}`"
))


## Shared tables

Window scores, negative-binomial quantiles, spatial quantiles, and the log-likelihood caches are aggregated here. Later cells only draw.


In [ ]:
_MAGNITUDE_DENSITY = {"etas": "etas_likelihood", "FINE": "fine_likelihood"}


def scores_for(spec: dict) -> pd.DataFrame:
    return scores_all.loc[np.isclose(scores_all["horizon_days"], spec["horizon_days"])].copy()


def _read_table(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    for column in ("forecast_start", "forecast_end", "time"):
        if column in frame.columns:
            frame[column] = pd.to_datetime(frame[column], format="mixed")
    return frame


def _band_fraction(values) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return float("nan")
    return float(((finite >= 0.025) & (finite <= 0.975)).mean())


def _ks_uniform(values) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return float("nan")
    return float(scipy_stats.kstest(finite, "uniform").statistic)


def _mean_finite(values) -> float:
    kept = np.asarray(values, dtype=float)
    kept = kept[np.isfinite(kept)]
    if kept.size == 0:
        return float("nan")
    return float(kept.mean())


def _mean_log_density(density) -> float:
    values = np.log(np.asarray(density, dtype=float))
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    return float(values.mean())


def _sum_log_density(density) -> float:
    values = np.log(np.asarray(density, dtype=float))
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    return float(values.sum())


def _spacetime_ll_path(horizon_dir: Path) -> Path | None:
    for candidate in (
        horizon_dir / "analysis_cache" / "spacetime_loglikelihood.csv",
        horizon_dir / "spacetime_loglikelihood.csv",
    ):
        if candidate.is_file():
            return candidate
    return None


band_rows = []
if len(scores_all) and "poisson_delta2" in scores_all.columns:
    for (days, method), group in scores_all.groupby(["horizon_days", "method"], sort=False):
        band_rows.append({
            "horizon_days": float(days),
            "method": method,
            "ks_delta": _ks_uniform(group["poisson_delta2"]),
        })
band_fast = pd.DataFrame(band_rows)

nbd_rows = []
nbd_slide = pd.DataFrame()
for spec in specs:
    tests_path = spec["horizon_dir"] / "analysis_cache" / "window_tests.csv"
    if not tests_path.is_file():
        continue
    tests = _read_table(tests_path)
    if "nbd_delta2" not in tests.columns:
        continue
    for method, group in tests.groupby("method", sort=False):
        nbd_rows.append({
            "horizon_days": float(spec["horizon_days"]),
            "method": method,
            "ks_nbd": _ks_uniform(group["nbd_delta2"]),
        })
    if SLIDE_SPEC is not None and np.isclose(spec["horizon_days"], SLIDE_HORIZON):
        nbd_slide = tests.copy()
nbd_ks = pd.DataFrame(nbd_rows)

spatial_rows = []
spatial_slide = pd.DataFrame()
for spec in specs:
    path = spec["horizon_dir"] / "analysis_cache" / "presentation_tables" / "s_deltas.csv"
    if not path.is_file():
        continue
    spatial = _read_table(path)
    for method, group in spatial.groupby("method", sort=False):
        row = {
            "horizon_days": float(spec["horizon_days"]),
            "method": method,
        }
        if "zeta" in group.columns:
            row["ks_zeta"] = _ks_uniform(group["zeta"])
        if "binary_zeta" in group.columns:
            row["ks_binary_zeta"] = _ks_uniform(group["binary_zeta"])
        spatial_rows.append(row)
    if SLIDE_SPEC is not None and np.isclose(spec["horizon_days"], SLIDE_HORIZON):
        spatial_slide = spatial.copy()
spatial_band = pd.DataFrame(spatial_rows)

igpa_rows = []
if len(scores_all) and "IGPA" in scores_all.columns:
    for (days, method), group in scores_all.groupby(["horizon_days", "method"], sort=False):
        igpa = pd.to_numeric(group["IGPA"], errors="coerce").dropna()
        if igpa.empty:
            continue
        igpa_rows.append({
            "horizon_days": float(days),
            "method": method,
            "mean_igpa": float(igpa.mean()),
            "std_igpa": float(igpa.std(ddof=1)) if igpa.size > 1 else 0.0,
        })
igpa_table = pd.DataFrame(igpa_rows)

number_rows = []
if SLIDE_SPEC is not None:
    slide_scores = scores_for(SLIDE_SPEC)
    counts_path = SLIDE_SPEC["horizon_dir"] / "analysis_cache" / "window_realization_counts.csv"
    counts = pd.read_csv(counts_path) if counts_path.is_file() else pd.DataFrame()
    observed_total = float(slide_scores.drop_duplicates("step_index")["n_observed"].sum())
    for method in METHODS:
        group = slide_scores.loc[slide_scores["method"] == method]
        if group.empty:
            continue
        n_forecast = float(group["n_forecast"].sum())
        poisson_low, poisson_high = bayona_evaluations.fmd_poisson_intervals(np.array([n_forecast]))
        variance = float("nan")
        if len(counts) and "n_events" in counts.columns:
            sizes = (
                counts.loc[counts["method"] == method]
                .groupby("seed")["n_events"]
                .sum()
                .to_numpy(dtype=float)
            )
            if sizes.size >= 2:
                variance = float(np.var(sizes, ddof=1))
        nbd = bayona_evaluations.negative_binomial_interval(n_forecast, variance)
        number_rows.append({
            "horizon_days": SLIDE_HORIZON,
            "method": method,
            "label": figures.method_label(method),
            "n_forecast": n_forecast,
            "n_observed": observed_total,
            "delta_percent": bayona_evaluations.count_discrepancy(n_forecast, observed_total),
            "poisson_low": float(poisson_low[0]),
            "poisson_high": float(poisson_high[0]),
            "nbd_low": np.nan if nbd is None else float(nbd[0]),
            "nbd_high": np.nan if nbd is None else float(nbd[1]),
        })
cumulative_number = pd.DataFrame(number_rows)


def _magnitude_horizon_row(horizon_dir: Path, days: float) -> list[dict]:
    csv_path = horizon_dir / "analysis_cache" / "magnitude_likelihoods.csv"
    if not csv_path.is_file():
        return []
    frame = pd.read_csv(csv_path)
    observed = frame.loc[(frame["population"] == "test") & (frame["source"] == "observed")]
    forecasts = frame.loc[frame["population"] == "forecast"]
    rows = []
    for method in METHODS:
        column = _MAGNITUDE_DENSITY[method]
        per_seed = np.array([
            _mean_log_density(group[column])
            for _seed, group in forecasts.loc[forecasts["source"] == method].groupby("seed")
        ], dtype=float)
        per_seed = per_seed[np.isfinite(per_seed)]
        rows.append({
            "horizon_days": float(days),
            "method": method,
            "n_seeds": int(per_seed.size),
            "mean_log_likelihood": float(per_seed.mean()) if per_seed.size else np.nan,
            "std_log_likelihood": float(per_seed.std(ddof=1)) if per_seed.size > 1 else 0.0,
            "observed_test_log_likelihood": _mean_log_density(observed[column]),
            "test_mean_neg_log_likelihood": -_mean_log_density(observed[column]),
        })
    return rows


magnitude_rows = []
for spec in specs:
    magnitude_rows.extend(_magnitude_horizon_row(spec["horizon_dir"], spec["horizon_days"]))
magnitude_ll = pd.DataFrame(magnitude_rows)


def _joint_horizon_rows(horizon_dir: Path, days: float) -> list[dict]:
    spacetime_path = _spacetime_ll_path(horizon_dir)
    magnitude_path = horizon_dir / "analysis_cache" / "magnitude_likelihoods.csv"
    if spacetime_path is None or not magnitude_path.is_file():
        return []
    spacetime = pd.read_csv(spacetime_path, usecols=["kind", "method", "seed", "log_likelihood"])
    magnitude = pd.read_csv(
        magnitude_path,
        usecols=["population", "source", "seed", "fine_likelihood", "etas_likelihood"],
    )
    observed_values = spacetime.loc[spacetime["kind"] == "observed_test", "log_likelihood"].to_numpy(dtype=float)
    observed_values = observed_values[np.isfinite(observed_values)]
    observed_spacetime = float(observed_values.mean()) if observed_values.size else float("nan")
    observed_spacetime_sum = float(observed_values.sum()) if observed_values.size else float("nan")
    observed_magnitude = magnitude.loc[
        (magnitude["population"] == "test") & (magnitude["source"] == "observed")
    ]
    forecast_spacetime = spacetime.loc[spacetime["kind"] == "forecast"]
    forecast_magnitude = magnitude.loc[magnitude["population"] == "forecast"]
    rows = []
    for method in METHODS:
        column = _MAGNITUDE_DENSITY[method]
        spacetime_by_seed = {}
        spacetime_sum_by_seed = {}
        for seed, group in forecast_spacetime.loc[forecast_spacetime["method"] == method].groupby("seed"):
            values = group["log_likelihood"].to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            if values.size == 0:
                continue
            spacetime_by_seed[int(seed)] = float(values.mean())
            spacetime_sum_by_seed[int(seed)] = float(values.sum())
        magnitude_by_seed = {
            int(seed): _mean_log_density(group[column])
            for seed, group in forecast_magnitude.loc[forecast_magnitude["source"] == method].groupby("seed")
        }
        magnitude_sum_by_seed = {
            int(seed): _sum_log_density(group[column])
            for seed, group in forecast_magnitude.loc[forecast_magnitude["source"] == method].groupby("seed")
        }
        joint = np.array([
            spacetime_by_seed[seed] + magnitude_by_seed[seed]
            for seed in sorted(spacetime_by_seed)
            if np.isfinite(spacetime_by_seed[seed]) and np.isfinite(magnitude_by_seed.get(seed, np.nan))
        ], dtype=float)
        joint_sums = np.array([
            spacetime_sum_by_seed[seed] + magnitude_sum_by_seed.get(seed, np.nan)
            for seed in sorted(spacetime_sum_by_seed)
            if np.isfinite(spacetime_sum_by_seed[seed]) and np.isfinite(magnitude_sum_by_seed.get(seed, np.nan))
        ], dtype=float)
        spacetime_sums = np.array(list(spacetime_sum_by_seed.values()), dtype=float)
        rows.append({
            "horizon_days": float(days),
            "method": method,
            "n_seeds": int(joint.size),
            "mean_log_likelihood": float(joint.mean()) if joint.size else np.nan,
            "std_log_likelihood": float(joint.std(ddof=1)) if joint.size > 1 else 0.0,
            "observed_test_log_likelihood": (
                observed_spacetime + _mean_log_density(observed_magnitude[column])
            ),
            "spacetime_n_seeds": int(spacetime_sums.size),
            "spacetime_sum_mean": float(spacetime_sums.mean()) if spacetime_sums.size else np.nan,
            "spacetime_sum_std": float(spacetime_sums.std(ddof=1)) if spacetime_sums.size > 1 else 0.0,
            "spacetime_observed_sum": observed_spacetime_sum,
            "joint_sum_n_seeds": int(joint_sums.size),
            "joint_sum_mean": float(joint_sums.mean()) if joint_sums.size else np.nan,
            "joint_sum_std": float(joint_sums.std(ddof=1)) if joint_sums.size > 1 else 0.0,
            "observed_joint_sum": observed_spacetime_sum + _sum_log_density(observed_magnitude[column]),
        })
    return rows


joint_rows = []
for spec in specs:
    joint_rows.extend(_joint_horizon_rows(spec["horizon_dir"], spec["horizon_days"]))
joint_ll = pd.DataFrame(joint_rows)


def _stm_concat_horizon_rows(horizon_dir: Path, days: float) -> list[dict]:
    """Full-test-set STM log-likelihood from concatenated walk-forward windows.

    For each seed, windows are contiguous and non-overlapping, so the catalog
    spanning the test set has

        LL_STM = sum_w (sum_i log λ_w(i) - Λ_w) + sum_e log p(m_e)
               = sum_w LL_st(w) + sum_e log p(m_e)

    (each cached ``log_likelihood`` already subtracts that window's integral).
    The per-seed score is that total divided by the number of events in the
    concatenated catalog; the horizon summary is the mean over seeds.
    """
    spacetime_path = _spacetime_ll_path(horizon_dir)
    magnitude_path = horizon_dir / "analysis_cache" / "magnitude_likelihoods.csv"
    if spacetime_path is None or not magnitude_path.is_file():
        return []
    spacetime = pd.read_csv(
        spacetime_path,
        usecols=[
            "kind", "method", "seed", "log_likelihood", "n_events",
            "window_start", "window_end",
        ],
    )
    magnitude = pd.read_csv(
        magnitude_path,
        usecols=[
            "population", "source", "seed", "time_unix",
            "fine_likelihood", "etas_likelihood",
        ],
    )
    observed_st = spacetime.loc[spacetime["kind"] == "observed_test"]
    if observed_st.empty:
        return []
    window_start = pd.to_datetime(observed_st["window_start"], format="mixed")
    window_end = pd.to_datetime(observed_st["window_end"], format="mixed")
    t0 = int(window_start.min().value // 10**9)
    t1 = int(window_end.max().value // 10**9)
    observed_st_ll = float(
        pd.to_numeric(observed_st["log_likelihood"], errors="coerce").sum()
    )
    observed_n = int(pd.to_numeric(observed_st["n_events"], errors="coerce").fillna(0).sum())
    observed_magnitude = magnitude.loc[
        (magnitude["population"] == "test")
        & (magnitude["source"] == "observed")
        & (magnitude["time_unix"] >= t0)
        & (magnitude["time_unix"] < t1)
    ]
    forecast_st = spacetime.loc[spacetime["kind"] == "forecast"]
    forecast_magnitude = magnitude.loc[magnitude["population"] == "forecast"]
    rows = []
    for method in METHODS:
        column = _MAGNITUDE_DENSITY[method]
        per_seed = []
        for seed, group in forecast_st.loc[forecast_st["method"] == method].groupby("seed"):
            st_ll = pd.to_numeric(group["log_likelihood"], errors="coerce").to_numpy(dtype=float)
            st_ll = st_ll[np.isfinite(st_ll)]
            n_events = int(pd.to_numeric(group["n_events"], errors="coerce").fillna(0).sum())
            mag_group = forecast_magnitude.loc[
                (forecast_magnitude["source"] == method)
                & (forecast_magnitude["seed"] == int(seed))
            ]
            mag_ll = _sum_log_density(mag_group[column])
            if st_ll.size == 0 or n_events <= 0 or not np.isfinite(mag_ll):
                continue
            total = float(st_ll.sum()) + float(mag_ll)
            per_seed.append(total / float(n_events))
        per_seed = np.asarray(per_seed, dtype=float)
        observed_mag_ll = _sum_log_density(observed_magnitude[column])
        observed_total = observed_st_ll + observed_mag_ll
        observed_mean = (
            observed_total / float(observed_n)
            if observed_n > 0 and np.isfinite(observed_total)
            else float("nan")
        )
        rows.append({
            "horizon_days": float(days),
            "method": method,
            "n_seeds": int(per_seed.size),
            "n_observed_events": observed_n,
            "mean_log_likelihood": float(per_seed.mean()) if per_seed.size else np.nan,
            "std_log_likelihood": float(per_seed.std(ddof=1)) if per_seed.size > 1 else 0.0,
            "observed_test_log_likelihood": observed_mean,
        })
    return rows


stm_rows = []
for spec in specs:
    stm_rows.extend(_stm_concat_horizon_rows(spec["horizon_dir"], spec["horizon_days"]))
stm_ll = pd.DataFrame(stm_rows)

display(Markdown(
    f"Magnitude rows: {len(magnitude_ll)}. Joint rows: {len(joint_ll)}. "
    f"STM-concat rows: {len(stm_ll)}. "
    f"Spatial rows: {len(spatial_band)}. NBD rows: {len(nbd_ks)}."
))


## Cumulative counts

Non-overlapping windows at `SLIDE_HORIZON`.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


# Cumulative counts only. These numbers do not follow STYLE["font"].
CUMULATIVE_COUNTS_LINE_WIDTH = {"observed": 2.2, "etas": 3.2, "FINE": 3.2}
CUMULATIVE_COUNTS_LABELS = {
    "xlabel": "Walk-forward step",
    "ylabel": "Cumulative event count",
    "observed": "Observed",
    "etas": "ETAS expected",
    "FINE": "FINE expected",
}
CUMULATIVE_COUNTS_LABEL_SIZE = 18   # x and y axis titles
CUMULATIVE_COUNTS_TICK_SIZE = 16    # numeric tick labels
CUMULATIVE_COUNTS_LEGEND_SIZE = 11  # legend text
CUMULATIVE_COUNTS_ZOOM_STEP = None    # first window in the inset
CUMULATIVE_COUNTS_ZOOM_WINDOWS = None  # how many windows the inset shows

if SLIDE_SPEC is None:
    display(Markdown(f"*No cache for the {SLIDE_HORIZON:g}-day slide horizon.*"))
else:
    slide_scores = scores_for(SLIDE_SPEC)
    counts_path = SLIDE_SPEC["horizon_dir"] / "analysis_cache" / "window_realization_counts.csv"
    realizations = pd.read_csv(counts_path) if counts_path.is_file() else None
    if figures.windows_tile(slide_scores):
        _export(
            figures.fig_cumulative_counts(
                slide_scores,
                SLIDE_HORIZON,
                METHODS,
                line_width=CUMULATIVE_COUNTS_LINE_WIDTH,
                labels=CUMULATIVE_COUNTS_LABELS,
                label_size=CUMULATIVE_COUNTS_LABEL_SIZE,
                tick_size=CUMULATIVE_COUNTS_TICK_SIZE,
                legend_size=CUMULATIVE_COUNTS_LEGEND_SIZE,
                realizations=realizations,
                window_lines=True,
                zoom_step=CUMULATIVE_COUNTS_ZOOM_STEP,
                zoom_windows=CUMULATIVE_COUNTS_ZOOM_WINDOWS,
            ),
            "cumulative_counts",
            horizon_tag(SLIDE_HORIZON),
        )
    else:
        display(Markdown("*Skipped cumulative counts: the windows do not tile the test.*"))


## Magnitude log-likelihood

FINE uses the MAGNET density and ETAS uses the Gutenberg–Richter density. All panels plot mean \(-\log p(m)\): the test-set bars use observed test events; the all-realizations bars and the solid vs-horizon markers use each seed’s forecast events (marker = mean across seeds, band = sample std); dashed lines are the observed test score.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if len(magnitude_ll):
    slide_magnitude = magnitude_ll.loc[np.isclose(magnitude_ll["horizon_days"], SLIDE_HORIZON)]
    test_values = {
        str(row["method"]): float(row["test_mean_neg_log_likelihood"])
        for _index, row in slide_magnitude.iterrows()
    }
    _export(
        figures.fig_method_bars(
            test_values,
            METHODS,
            title="Test set",
            ylabel=r"mean $-\log p(m)$",
        ),
        "magnitude_test_set_neg_log_likelihood",
        horizon_tag(SLIDE_HORIZON),
    )
    own_order = ["FINE", "etas"]
    own_rows = slide_magnitude.set_index("method")
    own_values = {
        method: -float(own_rows.loc[method, "mean_log_likelihood"])
        for method in own_order
        if method in own_rows.index and np.isfinite(own_rows.loc[method, "mean_log_likelihood"])
    }
    n_fine = int(own_rows.loc["FINE", "n_seeds"]) if "FINE" in own_rows.index else 0
    n_etas = int(own_rows.loc["etas", "n_seeds"]) if "etas" in own_rows.index else 0
    _export(
        figures.fig_method_bars(
            own_values,
            own_order,
            title=f"All realizations ({n_fine} FINE, {n_etas} ETAS)",
            ylabel=r"mean $-\log p(m)$",
        ),
        "magnitude_all_realizations_neg_log_likelihood",
        horizon_tag(SLIDE_HORIZON),
    )
    magnitude_neg = magnitude_ll.copy()
    magnitude_neg["mean_log_likelihood"] = -magnitude_neg["mean_log_likelihood"]
    if "observed_test_log_likelihood" in magnitude_neg.columns:
        magnitude_neg["observed_test_log_likelihood"] = -magnitude_neg[
            "observed_test_log_likelihood"
        ]
    _export(
        figures.fig_loglikelihood_vs_horizon(
            magnitude_neg,
            HORIZON_LIST,
            METHODS,
            title="Magnitude log-likelihood vs horizon",
            ylabel=r"Mean $-\log p(m)$",
            observed="per_method",
        ),
        "magnitude_log_likelihood_vs_horizon",
        "across_horizons",
    )
else:
    display(Markdown("*No `magnitude_likelihoods.csv` in the cached horizons.*"))


## Joint catalog log-likelihood

Each point is the mean, across seeds, of that seed's mean space-time window log-likelihood plus its mean magnitude log-likelihood. The band is one standard deviation of those per-seed totals. At `SLIDE_HORIZON`, the own-realization bars are the negative sum of each seed's window scores: space-time alone, with the observed test as a horizontal line, and space-time plus magnitude beside the observed-test bar.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if len(joint_ll):
    slide_joint = joint_ll.loc[np.isclose(joint_ll["horizon_days"], SLIDE_HORIZON)]
    if len(slide_joint) and "spacetime_sum_mean" in slide_joint.columns:
        indexed = slide_joint.set_index("method")
        order = [method for method in ("FINE", "etas") if method in indexed.index]
        n_fine = int(indexed.loc["FINE", "spacetime_n_seeds"]) if "FINE" in indexed.index else 0
        n_etas = int(indexed.loc["etas", "spacetime_n_seeds"]) if "etas" in indexed.index else 0
        _export(
            figures.fig_method_bars(
                {method: -float(indexed.loc[method, "spacetime_sum_mean"]) for method in order},
                order,
                title=f"Own realizations ({n_fine} FINE, {n_etas} ETAS)",
                ylabel=r"$-$ mean space-time log-likelihood",
                yerr={method: float(indexed.loc[method, "spacetime_sum_std"]) for method in order},
                hline=-float(indexed.loc[order[0], "spacetime_observed_sum"]),
                hline_label="observed test",
                zoom=True,
            ),
            "spacetime_own_realizations",
            horizon_tag(SLIDE_HORIZON),
        )
        n_fine = int(indexed.loc["FINE", "joint_sum_n_seeds"]) if "FINE" in indexed.index else 0
        n_etas = int(indexed.loc["etas", "joint_sum_n_seeds"]) if "etas" in indexed.index else 0
        _export(
            figures.fig_grouped_method_bars(
                {method: -float(indexed.loc[method, "joint_sum_mean"]) for method in order},
                {method: -float(indexed.loc[method, "observed_joint_sum"]) for method in order},
                order,
                title=f"Own realizations ({n_fine} FINE, {n_etas} ETAS)",
                ylabel=r"$-$ mean $(LL_{\mathrm{spacetime}} + LL_{\mathrm{mag}})$",
                yerr={method: float(indexed.loc[method, "joint_sum_std"]) for method in order},
                zoom=True,
            ),
            "joint_own_realizations",
            horizon_tag(SLIDE_HORIZON),
        )
    _export(
        figures.fig_loglikelihood_vs_horizon(
            joint_ll,
            HORIZON_LIST,
            METHODS,
            title="Joint catalog log-likelihood vs horizon",
            ylabel=r"Mean of per-realization mean $(LL_{\mathrm{spacetime}} + LL_{\mathrm{mag}})$",
            observed="none",
            yscale="symlog",
        ),
        "joint_catalog_log_likelihood_vs_horizon",
        "across_horizons",
    )
else:
    display(Markdown("*No joint log-likelihood table. A horizon needs both likelihood caches.*"))


## Space–time–magnitude log-likelihood (concatenated test set)

For each realization, all walk-forward windows are treated as one catalog over the full test span. The score is the point-process STM log-likelihood

$$\mathrm{LL}_{\mathrm{STM}}
= \sum_i \log\bigl(\lambda(t_i,x_i,y_i)\,p^{(\mathrm{model})}(m_i)\bigr)
- \int\lambda\,dx\,dy\,dt
= \sum_w\mathrm{LL}_{\mathrm{st}}(w) + \sum_e\log p(m_e),$$

where each cached window \(\mathrm{LL}_{\mathrm{st}}\) already subtracts that window’s compensator. The per-realization value is \(\mathrm{LL}_{\mathrm{STM}}/N\) ( \(N\) = events in that concatenated catalog). Bars and curves plot the mean over seeds of \(-\mathrm{LL}_{\mathrm{STM}}/N\); the gray bar / dashed reference is the same quantity on the observed test catalog.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if len(stm_ll):
    slide_stm = stm_ll.loc[np.isclose(stm_ll["horizon_days"], SLIDE_HORIZON)]
    if len(slide_stm):
        indexed = slide_stm.set_index("method")
        order = [method for method in ("FINE", "etas") if method in indexed.index]
        n_fine = int(indexed.loc["FINE", "n_seeds"]) if "FINE" in indexed.index else 0
        n_etas = int(indexed.loc["etas", "n_seeds"]) if "etas" in indexed.index else 0
        _export(
            figures.fig_grouped_method_bars(
                {method: -float(indexed.loc[method, "mean_log_likelihood"]) for method in order},
                {
                    method: -float(indexed.loc[method, "observed_test_log_likelihood"])
                    for method in order
                },
                order,
                title=f"STM mean LL ({n_fine} FINE, {n_etas} ETAS)",
                ylabel=r"$-(\sum\log(\lambda p)-\Lambda)/N$",
                yerr={method: float(indexed.loc[method, "std_log_likelihood"]) for method in order},
                zoom=True,
                # ylim=(8.5, 10),
            ),
            "stm_concat_mean_ll_own_realizations",
            horizon_tag(SLIDE_HORIZON),
        )
    stm_neg = stm_ll.copy()
    stm_neg["mean_log_likelihood"] = -stm_neg["mean_log_likelihood"]
    if "observed_test_log_likelihood" in stm_neg.columns:
        stm_neg["observed_test_log_likelihood"] = -stm_neg["observed_test_log_likelihood"]
    _export(
        figures.fig_loglikelihood_vs_horizon(
            stm_neg,
            HORIZON_LIST,
            METHODS,
            title="STM mean log-likelihood vs horizon",
            ylabel=r"$-(\sum\log(\lambda p)-\Lambda)/N$",
            observed="per_method",
            yscale="symlog",
        ),
        "stm_concat_mean_ll_vs_horizon",
        "across_horizons",
    )
    _export(
        figures.fig_loglikelihood_vs_horizon(
            stm_neg,
            HORIZON_LIST,
            METHODS,
            title="STM mean log-likelihood vs horizon",
            ylabel=r"$-(\sum\log(\lambda p)-\Lambda)/N$",
            observed="per_method",
            # yscale="log",
            ylim=(5, 15.0),
        ),
        "stm_concat_mean_ll_vs_horizon_symlog_ylim",
        "across_horizons",
    )
else:
    display(Markdown(
        "*No STM-concat table. Each horizon needs spacetime and magnitude likelihood caches.*"
    ))


## Per-window STM and magnitude $-$mean LL

Unlike the concatenated STM section above, each walk-forward window is scored on its own (observed history up to $t_0$, synthetic events in $[t_0, t_0+H)$), then seed-averaged within the window and averaged over windows.

- **Bar at `SLIDE_HORIZON`:** FINE vs ETAS mean $-\overline{\mathrm{LL}}_{\mathrm{STM}}$ (y-label $\mathcal{L}$).
- **Vs horizon:** the same per-window $-\overline{\mathrm{LL}}_{\mathrm{STM}}$ and $-\overline{\log p(m)}$.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


_MAG_DENSITY = {"etas": "etas_likelihood", "FINE": "fine_likelihood"}


def _to_unix_seconds(series: pd.Series) -> np.ndarray:
    values = pd.to_datetime(series, format="mixed")
    return (values.to_numpy(dtype="datetime64[ns]").astype("int64") // 10**9).astype(np.int64)


def _mag_window_totals(times, density, starts, ends):
    left = np.searchsorted(times, starts, side="left")
    right = np.searchsorted(times, ends, side="left")
    n = (right - left).astype(np.int64)
    logp = np.log(np.asarray(density, dtype=float))
    logp = np.where(np.isfinite(logp), logp, 0.0)
    csum = np.concatenate([[0.0], np.cumsum(logp)])
    sums = csum[right] - csum[left]
    means = np.full(starts.shape, np.nan, dtype=float)
    positive = n > 0
    means[positive] = sums[positive] / n[positive].astype(float)
    sums = sums.astype(float)
    sums[~positive] = 0.0
    return n, sums, means


def _per_window_horizon_summaries(horizon_dir: Path, days: float) -> list[dict]:
    """Seed-averaged per-window means, then mean/std over windows (minus mean LL)."""
    cache = Path(horizon_dir) / "analysis_cache"
    st_path = cache / "spacetime_loglikelihood.csv"
    if not st_path.is_file():
        alt = Path(horizon_dir) / "spacetime_loglikelihood.csv"
        st_path = alt if alt.is_file() else st_path
    mag_path = cache / "magnitude_likelihoods.csv"
    if not st_path.is_file() or not mag_path.is_file():
        return []

    st = pd.read_csv(
        st_path,
        usecols=[
            "step_index", "kind", "method", "seed",
            "window_start", "window_end", "n_events", "log_likelihood",
        ],
    )
    mag = pd.read_csv(
        mag_path,
        usecols=[
            "population", "source", "seed", "time_unix",
            "fine_likelihood", "etas_likelihood",
        ],
    )
    forecast = st.loc[st["kind"] == "forecast"].copy()
    if forecast.empty:
        return []
    forecast["method"] = forecast["method"].astype(str)
    forecast["seed"] = forecast["seed"].astype(int)
    forecast["start_unix"] = _to_unix_seconds(forecast["window_start"])
    forecast["end_unix"] = _to_unix_seconds(forecast["window_end"])

    observed_st = st.loc[st["kind"] == "observed_test"].copy()
    observed_st["start_unix"] = _to_unix_seconds(observed_st["window_start"])
    observed_st["end_unix"] = _to_unix_seconds(observed_st["window_end"])
    obs_st_by_step = observed_st.set_index("step_index")

    forecast_mag = mag.loc[mag["population"] == "forecast"].copy()
    forecast_mag["source"] = forecast_mag["source"].astype(str)
    forecast_mag["seed"] = forecast_mag["seed"].astype(int)
    observed_mag = mag.loc[
        (mag["population"] == "test") & (mag["source"] == "observed")
    ].copy()

    mag_index = {}
    for (source, seed), group in forecast_mag.groupby(["source", "seed"], sort=False):
        times = group["time_unix"].to_numpy(dtype=np.int64)
        order = np.argsort(times, kind="mergesort")
        times = times[order]
        dens = {
            "fine_likelihood": group["fine_likelihood"].to_numpy(dtype=float)[order],
            "etas_likelihood": group["etas_likelihood"].to_numpy(dtype=float)[order],
        }
        mag_index[(str(source), int(seed))] = (times, dens)

    obs_times = observed_mag["time_unix"].to_numpy(dtype=np.int64)
    obs_order = np.argsort(obs_times, kind="mergesort")
    obs_times = obs_times[obs_order]
    obs_dens = {
        "fine_likelihood": observed_mag["fine_likelihood"].to_numpy(dtype=float)[obs_order],
        "etas_likelihood": observed_mag["etas_likelihood"].to_numpy(dtype=float)[obs_order],
    }

    row_parts = []
    for (method, seed), group in forecast.groupby(["method", "seed"], sort=False):
        method = str(method)
        seed = int(seed)
        column = _MAG_DENSITY[method]
        group = group.sort_values("step_index")
        starts = group["start_unix"].to_numpy(dtype=np.int64)
        ends = group["end_unix"].to_numpy(dtype=np.int64)
        n_events = group["n_events"].to_numpy(dtype=np.int64)
        st_ll = group["log_likelihood"].to_numpy(dtype=float)
        steps = group["step_index"].to_numpy(dtype=int)

        times, dens = mag_index.get((method, seed), (np.array([], dtype=np.int64), {}))
        if times.size:
            n_mag, mag_sum, mag_mean = _mag_window_totals(times, dens[column], starts, ends)
        else:
            n_mag = np.zeros(len(group), dtype=np.int64)
            mag_sum = np.zeros(len(group), dtype=float)
            mag_mean = np.full(len(group), np.nan)

        failed = (n_events > 0) & (n_mag == 0)
        mag_sum = mag_sum.astype(float)
        mag_mean = mag_mean.astype(float)
        mag_sum[failed] = np.nan
        mag_mean[failed] = np.nan

        stm = st_ll + mag_sum
        stm_mean = np.full(len(group), np.nan)
        ok = n_events > 0
        stm_mean[ok] = stm[ok] / n_events[ok].astype(float)

        obs_n, obs_mag_sum, obs_mag_mean = _mag_window_totals(
            obs_times, obs_dens[column], starts, ends,
        )
        obs_st_ll = obs_st_by_step.reindex(steps)["log_likelihood"].to_numpy(dtype=float)
        obs_n_st = np.nan_to_num(
            obs_st_by_step.reindex(steps)["n_events"].to_numpy(dtype=float), nan=0.0
        ).astype(np.int64)
        obs_failed = (obs_n_st > 0) & (obs_n == 0)
        obs_mag_sum = obs_mag_sum.astype(float)
        obs_mag_mean = obs_mag_mean.astype(float)
        obs_mag_sum[obs_failed] = np.nan
        obs_mag_mean[obs_failed] = np.nan
        obs_stm = obs_st_ll + obs_mag_sum
        obs_stm_mean = np.full(len(group), np.nan)
        obs_ok = obs_n_st > 0
        obs_stm_mean[obs_ok] = obs_stm[obs_ok] / obs_n_st[obs_ok].astype(float)

        row_parts.append(pd.DataFrame({
            "step_index": steps,
            "method": method,
            "seed": seed,
            "stm_mean": stm_mean,
            "mag_mean": mag_mean,
            "obs_stm_mean": obs_stm_mean,
            "obs_mag_mean": obs_mag_mean,
        }))

    if not row_parts:
        return []
    per_ws = pd.concat(row_parts, ignore_index=True)
    window_mean = (
        per_ws.groupby(["method", "step_index"], sort=True)
        .agg(
            stm_mean=("stm_mean", "mean"),
            mag_mean=("mag_mean", "mean"),
            obs_stm_mean=("obs_stm_mean", "first"),
            obs_mag_mean=("obs_mag_mean", "first"),
        )
        .reset_index()
    )

    rows = []
    for method, g in window_mean.groupby("method", sort=False):
        stm = -g["stm_mean"].to_numpy(dtype=float)
        stm = stm[np.isfinite(stm)]
        mag = -g["mag_mean"].to_numpy(dtype=float)
        mag = mag[np.isfinite(mag)]
        obs_stm = -g["obs_stm_mean"].to_numpy(dtype=float)
        obs_stm = obs_stm[np.isfinite(obs_stm)]
        obs_mag = -g["obs_mag_mean"].to_numpy(dtype=float)
        obs_mag = obs_mag[np.isfinite(obs_mag)]
        rows.append({
            "horizon_days": float(days),
            "method": str(method),
            "n_windows": int(stm.size),
            "stm_mean_log_likelihood": float(stm.mean()) if stm.size else np.nan,
            "stm_std_log_likelihood": float(stm.std(ddof=1)) if stm.size > 1 else 0.0,
            "stm_observed_test_log_likelihood": float(obs_stm.mean()) if obs_stm.size else np.nan,
            "mag_mean_log_likelihood": float(mag.mean()) if mag.size else np.nan,
            "mag_std_log_likelihood": float(mag.std(ddof=1)) if mag.size > 1 else 0.0,
            "mag_observed_test_log_likelihood": float(obs_mag.mean()) if obs_mag.size else np.nan,
        })
    return rows


per_window_rows = []
for spec in specs:
    per_window_rows.extend(
        _per_window_horizon_summaries(spec["horizon_dir"], spec["horizon_days"])
    )
per_window_mll = pd.DataFrame(per_window_rows)
display(Markdown(
    f"Per-window −mean LL rows: **{len(per_window_mll)}** "
    f"(horizons: {', '.join(f'{d:g} d' for d in sorted(per_window_mll['horizon_days'].unique())) if len(per_window_mll) else 'none'})."
))


def _fig_simple_method_bars(
    values: dict[str, float],
    *,
    title: str,
    observed: dict[str, float] | None = None,
    ylim: tuple[float, float] | None = None,
) -> plt.Figure | None:
    """FINE / ETAS bars with $\\mathcal{L}$ ylabel (slide reference style).

    ``ylim`` forces the y-axis limits when set; otherwise the axis is padded
    from a default floor around the bars (and observed dashed lines).
    """
    order = [m for m in ("FINE", "etas") if np.isfinite(values.get(m, np.nan))]
    if not order:
        return None
    fig, ax = plt.subplots(figsize=STYLE["figsize"]["hist"])
    xs = np.arange(len(order), dtype=float)
    width = 0.65
    heights = np.array([float(values[m]) for m in order], dtype=float)
    ax.bar(
        xs,
        heights,
        width=width,
        color=[figures.method_color(m) for m in order],
        edgecolor="none",
        zorder=2,
    )
    obs_heights = []
    if observed:
        for i, method in enumerate(order):
            y_obs = observed.get(method, np.nan)
            if not np.isfinite(y_obs):
                continue
            y_obs = float(y_obs)
            obs_heights.append(y_obs)
            ax.plot(
                [xs[i] - width / 2, xs[i] + width / 2],
                [y_obs, y_obs],
                color=figures.method_color(method),
                linestyle="--",
                linewidth=1.8,
                solid_capstyle="butt",
                zorder=3,
                label=f"{figures.method_label(method)} observed",
            )
    ax.set_xticks(xs, [figures.method_label(m) for m in order])
    ax.set_ylabel(r"$\mathcal{L}$", fontsize=STYLE["font"]["label"] + 4)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        finite = list(heights) + obs_heights
        ymax = float(np.nanmax(finite)) if finite else 1.0
        lower_y = 8.0
        ax.set_ylim(lower_y, ymax + 0.5 * abs(ymax - lower_y) if ymax > 0 else 1.0)
    from matplotlib.ticker import MultipleLocator
    ax.yaxis.set_major_locator(MultipleLocator(0.5))
    ax.grid(False)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if obs_heights:
        ax.legend(frameon=False, fontsize=STYLE["font"]["legend"])
    fig.tight_layout()
    return fig

if len(per_window_mll):
    slide = per_window_mll.loc[np.isclose(per_window_mll["horizon_days"], SLIDE_HORIZON)]
    if len(slide):
        values = {
            str(row["method"]): float(row["stm_mean_log_likelihood"])
            for _, row in slide.iterrows()
        }
        observed = {
            str(row["method"]): float(row["stm_observed_test_log_likelihood"])
            for _, row in slide.iterrows()
        }
        _export(
            _fig_simple_method_bars(
                values,
                title=f"Per-window STM $-$mean LL — T = {SLIDE_HORIZON:g} d",
                # observed=observed,
                ylim=(8.5, 10),
            ),
            "per_window_stm_bars",
            horizon_tag(SLIDE_HORIZON),
        )

    stm_vs = per_window_mll.rename(columns={
        "stm_mean_log_likelihood": "mean_log_likelihood",
        "stm_std_log_likelihood": "std_log_likelihood",
        "stm_observed_test_log_likelihood": "observed_test_log_likelihood",
    })
    mag_vs = per_window_mll.rename(columns={
        "mag_mean_log_likelihood": "mean_log_likelihood",
        "mag_std_log_likelihood": "std_log_likelihood",
        "mag_observed_test_log_likelihood": "observed_test_log_likelihood",
    })
    _export(
        figures.fig_loglikelihood_vs_horizon(
            stm_vs,
            HORIZON_LIST,
            METHODS,
            title="Per-window STM $-$mean LL vs horizon",
            ylabel=r"$-\overline{\mathrm{LL}}_{\mathrm{STM}}$",
            observed="per_method",
        ),
        "per_window_stm_neg_mean_ll_vs_horizon",
        "across_horizons",
    )
    _export(
        figures.fig_loglikelihood_vs_horizon(
            mag_vs,
            HORIZON_LIST,
            METHODS,
            title="Per-window magnitude $-$mean LL vs horizon",
            ylabel=r"$-\overline{\log p(m)}$",
            observed="per_method",
        ),
        "per_window_mag_neg_mean_ll_vs_horizon",
        "across_horizons",
    )
else:
    display(Markdown(
        "*No per-window −mean LL table. Need spacetime and magnitude likelihood caches.*"
    ))


## Cumulative information gain

The walk-forward curve is the running sum of FINE's information gain per alarm against ETAS. The horizon curve is the mean window IGPA, with a ±1 standard-deviation band.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if SLIDE_SPEC is not None:
    _export(
        figures.fig_cumulative_igpa(
            scores_for(SLIDE_SPEC),
            SLIDE_HORIZON,
            line_width=CUMULATIVE_COUNTS_LINE_WIDTH,
            labels=CUMULATIVE_COUNTS_LABELS,
            label_size=24,
            tick_size=20,
        ),
        "cumulative_information_gain",
        horizon_tag(SLIDE_HORIZON),
    )
if len(igpa_table):
    _export(
        figures.fig_metric_mean_std_vs_horizon(
            igpa_table,
            "mean_igpa",
            "std_igpa",
            HORIZON_LIST,
            title="Mean information gain per alarm (FINE vs ETAS)",
            ylabel="Mean IGPA",
            methods=["FINE"],
            hline=0.0,
        ),
        "mean_igpa",
        "across_horizons",
    )


## Number test

$\delta = F(N_{\mathrm{obs}} \mid N_{\mathrm{fore}})$ at `SLIDE_HORIZON`, then the KS distance of those quantiles from a uniform, and the fraction of windows inside the Poisson 95% interval.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if SLIDE_SPEC is not None and "poisson_delta2" in scores_all.columns:
    _export(
        figures.fig_quantile_series_combined(
            scores_for(SLIDE_SPEC),
            METHODS,
            "poisson_delta2",
            ylabel=r"$\delta$",
            title="N-test",
            horizon_days=SLIDE_HORIZON,
            tick_labelsize=14,
            labelsize=18,

        ),
        "n_test",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            scores_for(SLIDE_SPEC),
            METHODS,
            "poisson_delta2",
            xlabel=r"$\delta$",
            title="N-test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
        ),
        "n_test_histogram",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            scores_for(SLIDE_SPEC),
            METHODS,
            "poisson_delta2",
            xlabel=r"$\delta$",
            title="N-test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
            show_legend=False,
            show_summary=False,
        ),
        "n_test_histogram_plain",
        horizon_tag(SLIDE_HORIZON),
    )
if len(band_fast):
    _export(
        figures.fig_metric_vs_horizon(
            band_fast,
            "ks_delta",
            HORIZON_LIST,
            title="N-test δ KS statistic vs uniform",
            ylabel="KS statistic",
            methods=METHODS,
        ),
        "ks_delta",
        "across_horizons",
    )
if len(summary) and "fraction_poisson_consistent_95" in summary.columns:
    _export(
        figures.fig_metric_vs_horizon(
            summary,
            "fraction_poisson_consistent_95",
            HORIZON_LIST,
            title="Windows passing Poisson 95% number test",
            ylabel="Fraction of windows",
            methods=METHODS,
            ylim=(0.0, 1.0),
        ),
        "fraction_poisson_consistent_95",
        "across_horizons",
    )


## Negative-binomial number test

The series is the negative-binomial $\delta$ at `SLIDE_HORIZON`. The horizon curves are its KS statistic and the fraction of windows inside the negative-binomial 95% interval. The discrepancy is $\Delta = 100\,|N_{\mathrm{fore}} - N_{\mathrm{obs}}| / \max(N_{\mathrm{fore}}, N_{\mathrm{obs}})$.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if len(nbd_slide) and "nbd_delta2" in nbd_slide.columns:
    _export(
        figures.fig_quantile_series_combined(
            nbd_slide,
            METHODS,
            "nbd_delta2",
            ylabel=r"NBD $\delta$",
            title="Negative-binomial number test",
            horizon_days=SLIDE_HORIZON,
            tick_labelsize=14,
            labelsize=18,
        ),
        "nbd_number_test",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            nbd_slide,
            METHODS,
            "nbd_delta2",
            xlabel=r"NBD $\delta$",
            title="Negative-binomial number test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
        ),
        "nbd_number_test_histogram",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            nbd_slide,
            METHODS,
            "nbd_delta2",
            xlabel=r"NBD $\delta$",
            title="Negative-binomial number test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
            show_legend=False,
            show_summary=False,
        ),
        "nbd_number_test_histogram_plain",
        horizon_tag(SLIDE_HORIZON),
    )
if len(nbd_ks):
    _export(
        figures.fig_metric_vs_horizon(
            nbd_ks,
            "ks_nbd",
            HORIZON_LIST,
            title="NBD δ KS statistic vs uniform",
            ylabel="KS statistic",
            methods=METHODS,
        ),
        "ks_nbd",
        "across_horizons",
    )
if len(summary) and "fraction_nbd_consistent_95" in summary.columns:
    _export(
        figures.fig_metric_vs_horizon(
            summary,
            "fraction_nbd_consistent_95",
            HORIZON_LIST,
            title="Windows passing NBD 95% number test",
            ylabel="Fraction of windows",
            methods=METHODS,
            ylim=(0.0, 1.0),
        ),
        "fraction_nbd_consistent_95",
        "across_horizons",
    )
if len(summary) and "mean_delta_percent" in summary.columns:
    _export(
        figures.fig_metric_vs_horizon(
            summary,
            "mean_delta_percent",
            HORIZON_LIST,
            title="Mean per-window count discrepancy",
            ylabel="Discrepancy (%)",
            methods=METHODS,
        ),
        "mean_delta_percent",
        "across_horizons",
    )
if len(cumulative_number):
    _export(
        figures.fig_cumulative_number_rows(cumulative_number, SLIDE_HORIZON),
        "cumulative_number_test",
        horizon_tag(SLIDE_HORIZON),
    )


## Binary spatial test and S-test

Binary $\zeta$ and S-test $\zeta$ at `SLIDE_HORIZON`, then each quantile's KS distance from a uniform across horizons.


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if len(spatial_slide):
    _export(
        figures.fig_quantile_series_combined(
            spatial_slide,
            METHODS,
            "binary_zeta",
            ylabel=r"binary $\zeta$",
            title="Binary spatial test",
            horizon_days=SLIDE_HORIZON,
        ),
        "binary_spatial_test",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            spatial_slide,
            METHODS,
            "binary_zeta",
            xlabel=r"binary $\zeta$",
            title="Binary spatial test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
        ),
        "binary_spatial_test_histogram",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            spatial_slide,
            METHODS,
            "binary_zeta",
            xlabel=r"binary $\zeta$",
            title="Binary spatial test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
            show_legend=False,
            show_summary=False,
        ),
        "binary_spatial_test_histogram_plain",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_series_combined(
            spatial_slide,
            METHODS,
            "zeta",
            ylabel=r"$\zeta$",
            title="S-test",
            horizon_days=SLIDE_HORIZON,
        ),
        "s_test",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            spatial_slide,
            METHODS,
            "zeta",
            xlabel=r"$\zeta$",
            title="S-test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
        ),
        "s_test_histogram",
        horizon_tag(SLIDE_HORIZON),
    )
    _export(
        figures.fig_quantile_histogram_combined(
            spatial_slide,
            METHODS,
            "zeta",
            xlabel=r"$\zeta$",
            title="S-test",
            horizon_days=SLIDE_HORIZON,
            invert_x=True,
            show_legend=False,
            show_summary=False,
        ),
        "s_test_histogram_plain",
        horizon_tag(SLIDE_HORIZON),
    )
if len(spatial_band) and "ks_binary_zeta" in spatial_band.columns:
    _export(
        figures.fig_metric_vs_horizon(
            spatial_band,
            "ks_binary_zeta",
            HORIZON_LIST,
            title="Binary ζ KS statistic vs uniform",
            ylabel="KS statistic",
            methods=METHODS,
        ),
        "ks_binary_zeta",
        "across_horizons",
    )
if len(spatial_band) and "ks_zeta" in spatial_band.columns:
    _export(
        figures.fig_metric_vs_horizon(
            spatial_band,
            "ks_zeta",
            HORIZON_LIST,
            title="S-test ζ KS statistic vs uniform",
            ylabel="KS statistic",
            methods=METHODS,
        ),
        "ks_zeta",
        "across_horizons",
    )


## PL, N, M, and S tests

Each marker except the pseudolikelihood mean is one quantile for that horizon, computed on the full catalogs (about 30 realizations), not an average over forecast windows. The pseudolikelihood mean is the mean of those same full-catalog scores, and its band is their standard deviation. The quantile panels have no band: each horizon has one score.

- PL: mean and standard deviation of the full-catalog pseudolikelihoods
- PL \(\gamma_L\): Savran et al. (2020) equation 18 on those catalogs
- N: \(\min(\delta_1,\delta_2)\) of the catalog number test on those catalogs
- M: \(\kappa\) of the catalog magnitude test on those catalogs
- S: \(\zeta\) of the catalog spatial test on those catalogs
- Binary \(\zeta\): Bayona binary spatial quantile on the full test interval


In [ ]:
importlib.reload(figures)
figures.configure(STYLE)

# Set True to write this cell even when STYLE save_png, save_pdf, and save_svg are off.
resave_cell = False
if resave_cell:
    _export = lambda fig, filename, *folder: export(fig, filename, *folder, force=True)
else:
    _export = export


if not CSEP_MEAN_STD.is_file():
    display(Markdown(
        f"*No catalog-test cache at `{CSEP_MEAN_STD}`. "
        "Run the heavy cell in `notebooks/presentation_figures.ipynb` "
        "with `COMPUTE_HEAVY_ACROSS = True`.*"
    ))
else:
    csep_table = pd.read_csv(CSEP_MEAN_STD)
    if HORIZONS is not None and "horizon_days" in csep_table.columns:
        mask = np.isin(csep_table["horizon_days"].to_numpy(dtype=float), HORIZON_LIST)
        csep_table = csep_table.loc[mask].copy()
    mean_std_specs = [
        ("PL_mean", "PL_std", "PL-test mean pseudolikelihood vs forecast horizon", "Mean pseudolikelihood", "pl_test_mean_std"),
    ]
    for mean_column, std_column, title, ylabel, filename in mean_std_specs:
        if mean_column not in csep_table.columns or csep_table[mean_column].isna().all():
            display(Markdown(
                f"*Skipped `{title}`. The cache has no `{mean_column}`.*"
            ))
            continue
        _export(
            figures.fig_metric_mean_std_vs_horizon(
                csep_table,
                mean_column,
                std_column,
                HORIZON_LIST,
                title=title,
                ylabel=ylabel,
                methods=METHODS,
            ),
            filename,
            "across_horizons",
        )
    for column, title, ylabel, filename in (
        ("N_delta", r"N-test $\min(\delta_1,\delta_2)$ vs forecast horizon", r"$\min(\delta_1,\delta_2)$", "n_test_min_delta"),
        ("PL_delta", r"PL-test $\gamma_L$ vs forecast horizon", r"$\gamma_L$", "pl_test_gamma"),
        ("M_delta", r"M-test $\kappa$ vs forecast horizon", r"$\kappa$", "m_test_kappa"),
        ("S_delta", r"S-test $\zeta$ vs forecast horizon", r"$\zeta$", "s_test_zeta"),
    ):
        if column not in csep_table.columns or csep_table[column].isna().all():
            display(Markdown(f"*Skipped `{title}`. The cache has no `{column}`.*"))
            continue
        _export(
            figures.fig_metric_vs_horizon(
                csep_table,
                column,
                HORIZON_LIST,
                title=title,
                ylabel=ylabel,
                methods=METHODS,
                ylim=(0.0, 1.0),
            ),
            filename,
            "across_horizons",
        )
binary_path = CSEP_MEAN_STD.with_name("binary_zeta.csv")
if binary_path.is_file():
    binary_table = pd.read_csv(binary_path)
    if HORIZONS is not None and "horizon_days" in binary_table.columns:
        mask = np.isin(binary_table["horizon_days"].to_numpy(dtype=float), HORIZON_LIST)
        binary_table = binary_table.loc[mask].copy()
    _export(
        figures.fig_metric_vs_horizon(
            binary_table,
            "binary_zeta",
            HORIZON_LIST,
            title=r"Binary $\zeta$ vs forecast horizon",
            ylabel=r"Binary $\zeta$",
            methods=METHODS,
            ylim=(0.0, 1.0),
        ),
        "binary_zeta",
        "across_horizons",
    )
else:
    display(Markdown(
        f"*Skipped binary $\\zeta$. No full-interval scores at `{binary_path}`.*"
    ))
